# Cleaning the CatalogIQ Target Columns

The goal here is simple: clean obvious target-label problems without forcing the data into a taxonomy we have not actually proven.

The rule throughout this notebook is: **if a correction is deterministic, make it. If it is uncertain, leave it missing or flag it instead of guessing.**


## 1. Load the data and keep every decision traceable

Before changing anything, I attach the dataset name, a SHA-256 fingerprint of the source file, and the original row number. That gives us a stable way to trace review flags and corrections back to the supplied CSV.


In [100]:
from catalogiq import ROOT_PATH, TRAIN_PATH, TARGET_PATH, PROVIDED_DATA_PATH
from pathlib import Path
import csv
import pandas as pd
import re
import hashlib

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

TARGET_COLUMNS = [
    "Mnfr",
    "Brand",
    "Platform",
    "Segment",
    "Sub-Segment",
    "TargetAgeGroup",
]

print(ROOT_PATH)

c:\Users\sebas\PycharmProjects\Git\Critical-Materials-Recovery


In [101]:
def generate_sha256(path):

    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()

In [102]:
train = pd.read_csv(TRAIN_PATH)
target = pd.read_csv(TARGET_PATH)


for df in [train,target]:

    if df is train:
        dataset = "Selfcare_Training_data (1).csv"

    if df is target:
            dataset = "Selfcare_Target_data (1).csv"

    df["dataset"] = dataset
    df["source_sha256"] = generate_sha256(PROVIDED_DATA_PATH/f"{dataset}")
    df["source_row"] = df.index
    display(df.head())

print("Training shape:", train.shape)
print("Target shape:  ", target.shape)

C:\Users\sebas\AppData\Local\Temp\ipykernel_25760\1211670601.py:1: DtypeWarning: Columns (10: Sun1, 11: Sun2, 12: Sun3, 13: Sun4, 14: Sun5, 15: Notes) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(TRAIN_PATH)
C:\Users\sebas\AppData\Local\Temp\ipykernel_25760\1211670601.py:2: DtypeWarning: Columns (11: Sun2, 12: Sun3, 13: Sun4, 14: Sun5, 15: Notes) have mixed types. Specify dtype option on import or set low_memory=False.
  target = pd.read_csv(TARGET_PATH)


,Exclude,JoiningKey,Retailer,Category,Mnfr,Brand,Platform,Segment,Sub-Segment,TargetAgeGroup,Sun1,Sun2,Sun3,Sun4,Sun5,Notes,ProductCategory,ProductBrand,ProductName,ProductRating,ProductReviewsCount,XRatXRev,ReviewsCount,Sku,Upc,ProductModelNumber,ProductUrl,ProductImageUrl,MDM_InsertDateTime,MDM_Id,ProductDescription,ProductContents,dataset,source_sha256,source_row
0,NaN,00014b3db41adfe3911aefbfa88b05af,Amazon,Self Care,All others,UnItemised brand,NaN,External Analgesics,Creams/Gels & Medicated Patches,Adult,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Health Care > Over-the-Counter Medication > Pain Reliev...,Cora,Cora Period Menstrual Cramp Relief Kit Period Balm with Soothing Warmth (1 o...,3.9,50,15.6,4,B08QDPHXJF,NaN,NaN,https://www.amazon.com/dp/B08QDPHXJF,https://m.media-amazon.com/images/I/71XzvQ+vftL._AC_SL1500_.jpg | https://m....,45117.83254,1136754,NaN,INTUITIVELY DESIGNED. Soothe cramps where you need it most. The patch adhere...,Selfcare_Training_data (1).csv,00f1fcd0d45bc4196ea8c0e9fffa63897c8e24d12a7639468928fbd6afff4452,0
1,NaN,00015d42f7052a8d5c9d3489d1ac5a63,Amazon,Self Care,All others,UnItemised brand,NaN,Internal Analgesics,Speciality Pain,Adult,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Health Care > OTC Medications & Treatments > Pain Relie...,WellnessPartners,UTI Slip D-Mannose Non GMO Organic Source Veggie Capsules 90 Count (500 Mg),4.5,50,9,2,B00528Z60O,NaN,NaN,https://www.amazon.com/dp/B00528Z60O,https://m.media-amazon.com/images/I/71vOnIuiyjL._AC_SL1500_.jpg | https://m....,44936,291986,NaN,NaN,Selfcare_Training_data (1).csv,00f1fcd0d45bc4196ea8c0e9fffa63897c8e24d12a7639468928fbd6afff4452,1
2,NaN,000220380ac4fcd79388ecf4fe26488a,Amazon,Self Care,NaN,UnItemised brand,NaN,"Vitamins, Minerals & Supplements",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Health & Household > Vitamins, Minerals & Supplements > Herbal Supplements >...",Ellie's Best,Ellie's Best Lion’s Mane Mushroom Powder Extract Supplement - USDA Organic -...,3.6,5,7.2,2,B0B3NGHTWD,NaN,NaN,https://www.amazon.com/dp/B0B3NGHTWD,https://m.media-amazon.com/images/I/81m4gRGGVkL._AC_SL1500_.jpg | https://m....,45177.76383,2506345,NaN,100% USDA CERTIFIED ORGANIC Lion's Mane Mushroom Extract. DOUBLE WATER EXTRA...,Selfcare_Training_data (1).csv,00f1fcd0d45bc4196ea8c0e9fffa63897c8e24d12a7639468928fbd6afff4452,2
3,NaN,0003d86e9f1502b501aabd81294cfbc1,Walmart,Self Care,All others,Centrum,NaN,"Vitamins, Minerals & Supplements",Multivitamins,Adult,NaN,NaN,NaN,NaN,NaN,NaN,Health and Medicine>Vitamins and Supplements>Multivitamins>50 Plus Multivita...,Centrum,"Centrum Silver Multivitamin for Adults 50 Plus, Multimineral Supplement, 150 Ct",4.7,46,319.6,68,893202,3.00054E+11,4177-59,https://www.walmart.com/ip/Centrum-Silver-Multivitamin-for-Adults-50-Plus-Mu...,https://i5.walmartimages.com/seo/Centrum-Silver-Multivitamin-for-Adults-50-P...,44936,332923,NaN,NaN,Selfcare_Training_data (1).csv,00f1fcd0d45bc4196ea8c0e9fffa63897c8e24d12a7639468928fbd6afff4452,3
4,NaN,000515b579fa1b8bc359d6c48569ae63,Amazon,Self Care,All others,UnItemised brand,NaN,Internal Analgesics,Speciality Pain,Adult,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Health Care > OTC Medications & Treatments > Pain Relie...,Counterpain,Counterpain Muscular Pain Relief Hot Warm Balm 120g.,4.5,9,31.5,7,B00A1246ES,NaN,NaN,https://www.amazon.com/dp/B00A1246ES,https://m.media-amazon.com/images/I/71562Xx9MwS._AC_SL1500_.jpg | https://m....,44936,294779,NaN,NaN,Selfcare_Training_data (1).csv,00f1fcd0d45bc4196ea8c0e9fffa63897c8e24d12a7639468928fbd6afff4452,4


,Exclude,JoiningKey,Retailer,Category,Mnfr,Brand,Platform,Segment,Sub-Segment,TargetAgeGroup,Sun1,Sun2,Sun3,Sun4,Sun5,Notes,ProductCategory,ProductBrand,ProductName,ProductRating,ProductReviewsCount,XRatXRev,ReviewsCount,Sku,Upc,ProductModelNumber,ProductUrl,ProductImageUrl,MDM_InsertDateTime,MDM_Id,ProductDescription,ProductContents,dataset,source_sha256,source_row
0,NaN,000309b7492984f5f63b75a7fd898738,Amazon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Health & Household > Vitamins, Minerals & Supplements > Digestive Supplement...",Swanson,Swanson Prebiotic + Probiotic Fiber - Natural Supplement Promoting Digestive...,5,1,5,1,B0BCMTGNY6,NaN,NaN,https://www.amazon.com/dp/B0BCMTGNY6,https://m.media-amazon.com/images/I/71Tyrt2VlrL._AC_SL1200_.jpg | https://m....,45117.83254,1136755,NaN,DIGESTIVE HEALTH Support: Swanson's premium Probiotic + Prebiotic supplement...,Selfcare_Target_data (1).csv,0122a49d6e1ca5f4f0fb7a2ab381418bf0e63d9c5dc3e915d64e259a973951db,0
1,NaN,0005475457eac31e1b48113d32596657,Amazon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Oral Care > Oral Pain Relief > Emergency Dental Care,Topicy,"Tooth Repair Kit, Temporary Teeth Replacement Kit for Temporary Restoration ...",4.7,146,47,10,B0C5M7G2LS,NaN,NaN,https://www.amazon.com/dp/B0C5M7G2LS,https://m.media-amazon.com/images/I/71p78eoD5bL._AC_SL1500_.jpg | https://m....,45145.83061,1258179,NaN,[Safe & High Quality] The dental restoration kit uses thermoformed beads mad...,Selfcare_Target_data (1).csv,0122a49d6e1ca5f4f0fb7a2ab381418bf0e63d9c5dc3e915d64e259a973951db,1
2,NaN,0007449da680ef6267cdad691f882fc9,Amazon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Health & Household > Vitamins, Minerals & Supplements > Blended Vitamin & Mi...",MICHAEL'S,Michael's Naturopathic Programs Kidney Factors - 120 Vegetarian Tablets - Nu...,4.6,42,41.4,9,B0009RC4SU,NaN,NaN,https://www.amazon.com/dp/B0009RC4SU,https://m.media-amazon.com/images/I/61PffVGNF2L._AC_SL1500_.jpg | https://m....,44936,286907,NaN,NaN,Selfcare_Target_data (1).csv,0122a49d6e1ca5f4f0fb7a2ab381418bf0e63d9c5dc3e915d64e259a973951db,2
3,NaN,000989c388fee65a67a1b438cdcab1a0,Amazon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Health & Household > Vitamins, Minerals & Supplements > Herbal Supplements >...",Micro Ingredients,"Sustainably US Grown, Organic Barley Grass Powder, 20 Ounce (1.25 Pounds), R...",4.5,136,63,14,B01MT7T5XX,NaN,NaN,https://www.amazon.com/dp/B01MT7T5XX,https://m.media-amazon.com/images/I/614RPd5mQRL._AC_SL1366_.jpg | https://m....,44936,307895,NaN,NaN,Selfcare_Target_data (1).csv,0122a49d6e1ca5f4f0fb7a2ab381418bf0e63d9c5dc3e915d64e259a973951db,3
4,NaN,0011ea8dbe27bb276b2f8aec9a1cd142,Amazon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Health & Household > Vitamins, Minerals & Supplements > Essential Fatty Acid...",Toplux,Toplux Keto Pills Ketosis Diet - Natural Ketosis Using Ketone & Ketogenic Di...,3.5,1100,77,22,B094VKZZ5L,NaN,NaN,https://www.amazon.com/dp/B094VKZZ5L,https://m.media-amazon.com/images/I/81RRYFdp+bL._AC_SL1500_.jpg | https://m....,44967,372476,NaN,NaN,Selfcare_Target_data (1).csv,0122a49d6e1ca5f4f0fb7a2ab381418bf0e63d9c5dc3e915d64e259a973951db,4


Training shape: (84552, 35)
Target shape:   (28082, 35)


## 2. Quick target-column check

The training file contains the labels we need to clean. The target file is mostly blank in those same columns because those are the values we eventually need to predict.

I start by looking at the six classification columns and their loaded dtypes before making any changes.


In [103]:
train_subset = train[TARGET_COLUMNS]
target_subset = target[TARGET_COLUMNS]

print("TRAINING PREVIEW")
display(train_subset.head())

print("TARGET PREVIEW")
display(target_subset.head())

TRAINING PREVIEW


,Mnfr,Brand,Platform,Segment,Sub-Segment,TargetAgeGroup
0,All others,UnItemised brand,NaN,External Analgesics,Creams/Gels & Medicated Patches,Adult
1,All others,UnItemised brand,NaN,Internal Analgesics,Speciality Pain,Adult
2,NaN,UnItemised brand,NaN,"Vitamins, Minerals & Supplements",NaN,NaN
3,All others,Centrum,NaN,"Vitamins, Minerals & Supplements",Multivitamins,Adult
4,All others,UnItemised brand,NaN,Internal Analgesics,Speciality Pain,Adult


TARGET PREVIEW


,Mnfr,Brand,Platform,Segment,Sub-Segment,TargetAgeGroup
0,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN


In [104]:
print("TRAINING DATA TYPES")
display(train_subset[TARGET_COLUMNS].dtypes.rename("dtype").to_frame())

print("TARGET DATA TYPES")
display(target_subset[TARGET_COLUMNS].dtypes.rename("dtype").to_frame())

TRAINING DATA TYPES


,dtype
Mnfr,str
Brand,str
Platform,str
Segment,str
Sub-Segment,str
TargetAgeGroup,str


TARGET DATA TYPES


,dtype
Mnfr,float64
Brand,float64
Platform,float64
Segment,float64
Sub-Segment,float64
TargetAgeGroup,float64


## 3. First pass: catch obvious malformed target rows

Some bad rows are not normal taxonomy mistakes at all—they look like source values shifted into the wrong columns. The first pass looks for obvious warning signs such as strange label formatting, extreme label behavior, unusual row-level missingness/text length, and rows that lose expected numeric content.

Reason 2 (label formatting) is review-only and does not remove rows. Reason 3 remains review-only. Reasons 4, 5, and 6 quarantine rows, even if reason 2 is also present.


In [105]:
# Q_REASON checks are implemented in catalogiq.cleaning and used below.


In [106]:
def get_label_occurance(df:pd.DataFrame, col:str) -> pd.Series:

    current_col=df[col].astype("string")
    label_occurance = current_col.value_counts().sort_index()

    return label_occurance


In [107]:
def iqr_bounds(series, multiplier=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - (multiplier * iqr)
    upper = q3 + (multiplier * iqr)

    return lower, upper

In [108]:
def get_missingingness_bounds(df:pd.DataFrame):
    return iqr_bounds(df.isna().mean(axis=1))  


In [109]:
def get_label_len_bound(df:pd.DataFrame):
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    str_lengths = df[text_cols].apply(lambda col: col.astype("string").str.len())
    return iqr_bounds(str_lengths.mean(axis=1))

In [ ]:
from catalogiq.cleaning import clean_training

def check_target_cols(df: pd.DataFrame):
    audit = clean_training(df)
    # Keep supplied labels here; later cells explore the approved corrections.
    annotated = df.copy()
    all_rows = pd.concat([audit.cleaned, audit.quarantine]).reindex(df.index)
    annotated["Q_REASON"] = all_rows["Q_REASON"]
    return annotated.loc[audit.cleaned.index].copy(), annotated.loc[audit.quarantine.index].copy()

filtered_train, quarantined_rows = check_target_cols(train)
assert len(filtered_train) + len(quarantined_rows) == len(train)


In [ ]:
display(quarantined_rows.head())

In [ ]:
len(quarantined_rows)

In [ ]:
len(filtered_train)

In [ ]:
display(filtered_train)

## 4. Check whether Segment → Sub-Segment is really a strict hierarchy

A tempting rule is: *a child class cannot have more rows than its parent.* That only works when a Sub-Segment has exactly one parent.

The data shows that some Sub-Segments appear under more than one Segment, so I do **not** enforce a global one-parent hierarchy. Instead, the count check is limited to Sub-Segments that have only one observed parent.


In [ ]:
shared_subsegments = (
    filtered_train.dropna(subset=["Segment", "Sub-Segment"])
    .groupby("Sub-Segment")["Segment"]
    .nunique()
)

shared_names = shared_subsegments[shared_subsegments > 1].index

display(
    filtered_train.loc[
        filtered_train["Sub-Segment"].isin(shared_names),
        ["Segment", "Sub-Segment"]
    ]
    .drop_duplicates()
    .sort_values(["Sub-Segment", "Segment"])
)

In [ ]:
# How many rows each Segment has
parent_counts = filtered_train["Segment"].value_counts()

# How many total rows each Sub-Segment has
child_counts = filtered_train["Sub-Segment"].value_counts()

# Unique Segment/Sub-Segment relationships
relationships = (
    filtered_train.dropna(subset=["Segment", "Sub-Segment"])
    [["Segment", "Sub-Segment"]]
    .drop_duplicates()
)

# Number of parents each Sub-Segment belongs to
num_parents = (
    relationships
    .groupby("Sub-Segment")["Segment"]
    .nunique()
)

# Add counts
relationships["num_parents"] = relationships["Sub-Segment"].map(num_parents)
relationships["subsegment_rows"] = relationships["Sub-Segment"].map(child_counts)
relationships["parent_rows"] = relationships["Segment"].map(parent_counts)

# Does the child's TOTAL count exceed this parent's total count?
relationships["child_gt_parent"] = (
    relationships["subsegment_rows"] > relationships["parent_rows"]
)

display(
    relationships.sort_values(
        ["num_parents", "Sub-Segment"]
    )
)

In [ ]:
def check_single_parent_counts(
    df,
    parent_col="Segment",
    child_col="Sub-Segment"
):
    # only use complete parent-child pairs to determine relationships
    relationships = df.dropna(
        subset=[parent_col, child_col]
    )

    # number of observed parents for each child
    parent_counts_per_child = (
        relationships
        .groupby(child_col)[parent_col]
        .nunique()
    )

    # children with exactly one observed parent
    single_parent_children = (
        parent_counts_per_child[
            parent_counts_per_child == 1
        ].index
    )

    # find that single parent
    child_to_parent = (
        relationships[
            relationships[child_col].isin(single_parent_children)
        ]
        .groupby(child_col)[parent_col]
        .first()
    )

    # total counts across dataset
    child_counts = df[child_col].value_counts()
    parent_counts = df[parent_col].value_counts()

    results = []

    for child, parent in child_to_parent.items():

        child_rows = child_counts.get(child, 0)
        parent_rows = parent_counts.get(parent, 0)

        if child_rows > parent_rows:
            results.append({
                child_col: child,
                parent_col: parent,
                "child_rows": child_rows,
                "parent_rows": parent_rows,
                "difference": child_rows - parent_rows
            })

    return pd.DataFrame(results)

In [ ]:
violations = check_single_parent_counts(train)

display(violations)

### Decision

The single-parent check produced no violations. More importantly, the shared-parent cases show why a blanket “all child counts must be smaller than every parent” rule would be wrong.

I am keeping this as a structural sanity check, not turning the entire Segment/Sub-Segment relationship into a hard whitelist.


## 5. Normalize only label differences we can defend

Next I look for labels that collapse to the same value after harmless formatting cleanup. This is different from guessing that two semantically similar labels mean the same thing.


In [ ]:
def find_format_duplicates(df, col):
    labels = df[col].dropna().astype("string")

    def normalize_label(label):
        label = label.strip()
        label = re.sub(r"\s*/\s*", "/", label)
        label = re.sub(r"\s+", " ", label)
        return label.casefold()

    temp = pd.DataFrame({
        "original": labels,
        "normalized": labels.map(normalize_label)
    })

    groups = (
        temp.groupby("normalized")["original"]
        .agg(lambda x: sorted(set(x)))
    )

    # Only return normalized labels that came from
    # more than one original spelling
    return groups[groups.apply(len) > 1]

In [ ]:
for col in TARGET_COLUMNS:
    print(find_format_duplicates(filtered_train, col))

`Cold/Flu` and `Cold / Flu` differ only by spacing around `/`, so that is a safe formatting normalization.


In [ ]:
filtered_train["Sub-Segment"] = filtered_train["Sub-Segment"].replace({
    "Cold / Flu": "Cold/Flu"
})


### `Other Lifestyle` vs `Other Lifestyle CHC`

This one needed more than a string comparison. Both labels sit under `Lifestyle CHC`, but one has 195 rows and the other only 3. I checked whether the three smaller-class products duplicated products in the larger class by product name, UPC, SKU, or URL.


In [ ]:
filtered_train.loc[
    filtered_train["Sub-Segment"].isin([
        "Other Lifestyle CHC",
        "Other Lifestyle"
    ]),
    ["Segment", "Sub-Segment"]
].value_counts()

In [ ]:
small = filtered_train[filtered_train["Sub-Segment"] == "Other Lifestyle"]
large = filtered_train[filtered_train["Sub-Segment"] == "Other Lifestyle CHC"]

small_products = set(small["ProductName"].dropna())
large_products = set(large["ProductName"].dropna())

overlap = small_products & large_products

print("Matching products:", len(overlap))
display(overlap)

In [ ]:
for col in ["Upc", "Sku", "ProductUrl"]:
    small_values = set(small[col].dropna())
    large_values = set(large[col].dropna())

    overlap = small_values & large_values

    print(f"{col}: {len(overlap)} matches")
    print(overlap)

In [ ]:
display(
    filtered_train.loc[
        filtered_train["Sub-Segment"] == "Other Lifestyle",
        [
            "Segment",
            "Sub-Segment",
            "ProductCategory",
            "ProductName",
            "Brand"
        ]
    ]
)

In [ ]:
display(
    filtered_train.loc[
        filtered_train["Sub-Segment"] == "Other Lifestyle CHC",
        [
            "Segment",
            "Sub-Segment",
            "ProductCategory",
            "ProductName",
            "Brand"
        ]
    ].head(20)
)

There were no exact product duplicates across the two labels, so the decision is not based on duplicate records. After inspecting the three `Other Lifestyle` products, they did not show a clear separate taxonomy from `Other Lifestyle CHC`.

Because they share the same parent, the wording is nearly identical, and the three-row class does not show a distinct meaning, I normalize `Other Lifestyle` to `Other Lifestyle CHC`.


In [ ]:
filtered_train.loc[
    filtered_train["Sub-Segment"] == "Other Lifestyle",
    "Sub-Segment"
] = "Other Lifestyle CHC"


## 6. Validate `Mnfr` without inferring it from Brand

`Mnfr` has two accepted populated values: `J&J` and `All others`. Inspect Brand associations below, but preserve supplied manufacturer labels and missing values.


In [ ]:
brand_mnfr_counts = (
    filtered_train.dropna(subset=["Brand", "Mnfr"])
    .groupby("Brand")["Mnfr"]
    .nunique()
)

display(brand_mnfr_counts[brand_mnfr_counts > 1])

In [ ]:
display(
    filtered_train[
        filtered_train["Brand"].isin([
            "Benadryl",
            "Bengay",
            "Sudafed",
            "Tylenol"
        ])
    ]
    .groupby(["Brand", "Mnfr"])
    .size()
)

In [ ]:
check_cols = [
    "Brand",
    "ProductName",
    "ProductCategory",
    "Retailer"
]

duplicates_both_mnfr = (
    filtered_train[
        filtered_train["Brand"].isin([
            "Benadryl",
            "Bengay",
            "Sudafed",
            "Tylenol"
        ])
    ]
    .dropna(subset=check_cols + ["Mnfr"])
    .groupby(check_cols)["Mnfr"]
    .nunique()
)

display(
    duplicates_both_mnfr[
        duplicates_both_mnfr > 1
    ]
)

In [ ]:
brands = ["Benadryl", "Bengay", "Sudafed", "Tylenol"]

cols = [
    "ProductName",
    "ProductCategory",
    "Retailer"
]

subset = filtered_train[
    filtered_train["Brand"].isin(brands)
].dropna(subset=["Brand", "Mnfr"] + cols)

for col_to_ignore in cols:

    compare_cols = [
        "Brand",
        *[col for col in cols if col != col_to_ignore]
    ]

    conflicts = (
        subset
        .groupby(compare_cols)["Mnfr"]
        .nunique()
    )

    conflicts = conflicts[conflicts > 1]

    print(
        f"Ignoring {col_to_ignore}: "
        f"{len(conflicts)} cross-Mnfr matches"
    )

In [ ]:
brands = ["Benadryl", "Bengay", "Sudafed", "Tylenol"]

subset = filtered_train[
    filtered_train["Brand"].isin(brands)
].dropna(subset=["Brand", "Mnfr"])

for col in ["ProductName", "ProductCategory", "Retailer"]:

    conflicts = (
        subset
        .dropna(subset=[col])
        .groupby(["Brand", col])["Mnfr"]
        .nunique()
    )

    conflicts = conflicts[conflicts > 1]

    print(f"{col}: {len(conflicts)} shared across Mnfr")
    display(conflicts)

In [ ]:
display(
    filtered_train.loc[
        filtered_train["Brand"].isin(brands) &
        (filtered_train["Retailer"] == "Amazon")
    ]
    .groupby(["Brand", "Mnfr"])
    .agg(
        rows=("ProductName", "size"),
        products=("ProductName", "nunique"),
        categories=("ProductCategory", "nunique")
    )
)

In [ ]:
brands = ["Benadryl", "Bengay", "Sudafed", "Tylenol"]

exceptions = filtered_train[
    filtered_train["Brand"].isin(brands) &
    (filtered_train["Mnfr"] == "All others")
]

display(
    exceptions.T
)

In [ ]:
brand_mnfr_counts = (
    filtered_train.dropna(subset=["Brand", "Mnfr"])
    .groupby(["Brand", "Mnfr"])
    .size()
    .unstack(fill_value=0)
)

brand_mnfr_counts["total"] = brand_mnfr_counts.sum(axis=1)

brand_mnfr_counts["dominant_ratio"] = (
    brand_mnfr_counts[["J&J", "All others"]].max(axis=1)
    / brand_mnfr_counts["total"]
)

display(
    brand_mnfr_counts.loc[
        ["Benadryl", "Bengay", "Sudafed", "Tylenol"]
    ]
)

### Decision

Do not reassign manufacturer labels from Brand. A dominant association is not enough to establish that the minority labels are wrong, or that missing labels should be filled.

Preserve supplied `Mnfr` values. Flag populated values outside `J&J` and `All others` for review without replacing them.


In [ ]:
def clean_mnfr(df: pd.DataFrame):
    df = df.copy()

    valid_mnfr = {"J&J", "All others"}

    # flag anything outside the approved Mnfr values
    invalid_mask = (
        df["Mnfr"].notna()
        & ~df["Mnfr"].isin(valid_mnfr)
    )

    df.loc[invalid_mask, "Q_REASON"] = (
        df.loc[invalid_mask, "Q_REASON"]
        .apply(lambda x: sorted(set(x + [6])))
    )

    quarantined_rows = df.loc[invalid_mask].copy()

    return df, quarantined_rows

In [ ]:
filtered_mnfr_train, mnfr_review = clean_mnfr(filtered_train)

In [ ]:
len(mnfr_review)

In [ ]:
display(mnfr_review)

## 7. Investigate `Platform` instead of forcing a rule

Compare Platform availability using the supplied manufacturer labels. This is descriptive evidence only: manufacturer does not authorize a Platform fill.


In [ ]:
display(
    filtered_mnfr_train.groupby("Mnfr")["Platform"]
    .agg(
        rows="size",
        platform_labeled=lambda x: x.notna().sum(),
        platform_missing=lambda x: x.isna().sum()
    )
)

Inspect missing Platform among rows already labeled `Mnfr = J&J` in the source. Do not infer manufacturer membership from Brand.


In [ ]:
jnj_platform_blanks = filtered_mnfr_train[
    filtered_mnfr_train["Mnfr"].eq("J&J")
    & filtered_mnfr_train["Platform"].isna()
].copy()

print("Source-labeled J&J rows with missing Platform:", len(jnj_platform_blanks))


### Are those rows just missing Platform, or broadly incomplete?

Compare their missing-label rates with the full training set.


In [ ]:
compare_cols = [
    "Sub-Segment",
    "TargetAgeGroup",
    "Segment",
    "Brand",
    "Mnfr"
]

comparison = []

for col in compare_cols:
    subset_missing = jnj_platform_blanks[col].isna().mean()
    overall_missing = train[col].isna().mean()

    comparison.append({
        "column": col,
        "subset_missing_pct": subset_missing,
        "overall_missing_pct": overall_missing,
        "difference": subset_missing - overall_missing
    })

comparison_df = pd.DataFrame(comparison)

display(comparison_df)

Use the comparison above to describe missingness. Missing Platform stays missing; incomplete labels do not justify inferring either Platform or manufacturer.


## 8. Platform decision: do not backfill

Automatic Platform backfilling is omitted.

Although some missing Platform values have apparently matching labeled counterparts, the dataset does not contain a sufficiently reliable canonical product identifier. Repeated UPC, SKU, MDM_Id, and product-name matches can represent different products, variants, malformed values, or conflicting labels.

Because product identity cannot be validated confidently enough for automatic label transfer, missing Platform values stay missing rather than being inferred.


## Where this leaves the target cleaning

The useful rules from this notebook are intentionally narrow:

- Keep reason-2 formatting flags in the data; quarantine rows meeting reasons 4, 5, or 6.
- Do not assume every Sub-Segment has one Segment parent.
- Normalize deterministic formatting differences such as `Cold / Flu` → `Cold/Flu`.
- Normalize `Other Lifestyle` → `Other Lifestyle CHC` based on the documented review.
- Preserve supplied `Mnfr` values and validate only its two-value vocabulary; do not infer manufacturer from Brand.
- Do not force Platform from manufacturer alone.
- Do not automatically backfill Platform from UPC, SKU, MDM_Id, or product-name matches.
- Missing target labels stay missing unless a separate deterministic rule is established.

The separate ProductBrand → Brand investigation is intentionally left out of this notebook for now.

The main goal is to leave the dataset cleaner **without pretending we know more than the data actually supports**.
